# RL Course Project - Multi-Product Inventory Control

**Five qualifying techniques:** Tabular Q-Learning, REINFORCE with baseline, A2C, PPO, and Double DQN.

This notebook is the reproducibility record for configuration generation, shared preprocessing, model training, local validation, comparison, convergence evidence, and export of the five deterministic submission policies.

## 1. Reproducibility setup
Run this notebook from the project root containing `industrial_inventory_env/`, `src/`, `submissions/`, `train_all.py`, and `evaluate_all.py`.

In [1]:
from pathlib import Path
import json, shutil, importlib.util
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
ARTIFACT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)

Project root: C:\Users\rajsh\PycharmProjects\Industry_Inv_Control_RL
Torch: 2.13.0+cpu
NumPy: 2.5.2


## 2. Assigned parameter variant and configuration loading
The official environment deterministically maps the student's official roll number to the assigned variant. No environment parameters are manually altered.

In [2]:
from industrial_inventory_env import generate_student_config, public_config_summary

ROLL_NUMBER = input("Enter official roll number: ").strip()
student_config = generate_student_config(ROLL_NUMBER)
print(public_config_summary(student_config))
print("variant_id:", student_config.get("variant_id"))
print("config_fingerprint:", student_config.get("config_fingerprint"))

KeyboardInterrupt: Interrupted by user

**Report fields:** copy the printed `variant_id` and `config_fingerprint` into the final report before submission. The same `student_config` object is used for all five training pipelines and local validation.

## 3. Common observation preprocessing and action mapping
The neural methods use a 56-feature normalized state encoder. Derived inventory-control features include recency-weighted demand, inventory position, days of supply, lead-time demand, inventory gap, and next arrivals. Tabular Q-Learning uses a compact discretized state based on lead-time coverage, demand buckets, episode phase, and utilization.

In [ ]:
from src.common import (
    encode_observation, discretize_observation, demand_estimate,
    indices_to_quantities, quantities_to_indices, base_stock_fallback
)

# Mapping used by the environment: index 0..10 -> quantity 0..100 in steps of 10
example_indices = np.array([4, 2, 0])
print("Action indices:", example_indices)
print("Order quantities:", indices_to_quantities(example_indices))

# Submission run_policy() returns quantities, not indices.
assert indices_to_quantities([0, 10, 5]).tolist() == [0, 100, 50]

In [ ]:
# Inspect the encoder on one legal environment observation
from industrial_inventory_env import IndustrialInventoryEnv
env = IndustrialInventoryEnv(student_config, scenario_mode="stationary", domain_randomization=True)
obs, _ = env.reset(seed=123)
x = encode_observation(obs)
print("Encoded shape:", x.shape)
print("Demand estimate:", demand_estimate(obs))
print("Discretized tabular state:", discretize_observation(obs))
print("Fallback action indices:", base_stock_fallback(obs))

## 4. Technique-specific representations

- **Tabular Q-Learning:** per-state table of shape `3 x 11`; one 11-way action-value vector per product.
- **REINFORCE + baseline:** two 256-unit ReLU layers, 33 actor logits reshaped to `3 x 11`, plus scalar baseline.
- **A2C:** two 256-unit Tanh layers, 33 actor logits plus scalar critic.
- **PPO:** same actor-critic architecture as A2C, trained with clipped policy updates and GAE.
- **Double DQN:** two 256-unit ReLU layers with 33 outputs reshaped to three 11-action Q heads; online network selects, target network evaluates.

In [ ]:
from src.networks import ActorCritic, PolicyBaseline, FactorizedQNetwork

print("REINFORCE network:
", PolicyBaseline(hidden=256))
print("
A2C/PPO network:
", ActorCritic(hidden=256))
print("
Double DQN network:
", FactorizedQNetwork(hidden=256))

## 5. Reward used during training
The official environment reward is `-daily_cost / 100`. All five training implementations apply an additional constant factor `0.1` for numerical conditioning: `r_train = 0.1 * r_env`. This positive rescaling does not change which policy is optimal. Evaluation always uses the environment's original unscaled episode cost.

## 6. Main hyperparameters

In [ ]:
hyperparams = pd.DataFrame([
    ["Tabular Q-Learning", "12,000 episodes", "alpha=0.08; gamma=0.98; eps 1.0->0.03", 11],
    ["REINFORCE + baseline", "8,000 episodes", "gamma=0.99; lr=2e-4; entropy=0.005; hidden=256", 21],
    ["A2C", "600,000 steps", "n_steps=32; gamma=0.99; lr=2e-4; value=0.5; entropy=0.005", 31],
    ["PPO", "1,000,000 steps", "rollout=2048; epochs=10; batch=256; gamma=0.99; GAE=0.95; clip=0.2; lr=1e-4", 41],
    ["Double DQN", "700,000 steps", "batch=256; gamma=0.99; lr=2e-4; replay=200k; target_update=2500", 51],
], columns=["Technique","Training budget","Important hyperparameters","Seed"])
hyperparams

## 7. Reproducible training pipelines
Set `RUN_FULL_TRAINING=True` for the final reproducibility run. The same official `student_config` is passed to every trainer. Trained artifacts are saved under `artifacts/` and copied next to the corresponding submission policy.

In [ ]:
RUN_FULL_TRAINING = False  # set True for the final reproducibility run

from src.tabular_q import train as train_tabq, save_table
from src.reinforce import train as train_reinforce
from src.a2c import train as train_a2c
from src.ppo import train as train_ppo
from src.double_dqn import train as train_ddqn

if RUN_FULL_TRAINING:
    q = train_tabq(student_config, episodes=12_000)
    save_table(q, ARTIFACT_DIR / "tabular_q.npz")

    net = train_reinforce(student_config, episodes=8_000)
    torch.save(net.state_dict(), ARTIFACT_DIR / "reinforce.pt")

    net = train_a2c(student_config, total_steps=600_000)
    torch.save(net.state_dict(), ARTIFACT_DIR / "a2c.pt")

    net = train_ppo(student_config, total_steps=1_000_000, rollout_steps=2048)
    torch.save(net.state_dict(), ARTIFACT_DIR / "ppo.pt")

    net = train_ddqn(student_config, total_steps=700_000, learning_starts=10_000)
    torch.save(net.state_dict(), ARTIFACT_DIR / "double_dqn.pt")

    for name in ["tabular_q.npz","reinforce.pt","a2c.pt","ppo.pt","double_dqn.pt"]:
        shutil.copy2(ARTIFACT_DIR / name, SUBMISSION_DIR / name)

    print("Training complete; artifacts copied to submissions/.")
else:
    print("Training skipped. Set RUN_FULL_TRAINING=True to reproduce final models.")

## 8. Deterministic local validation
The final frozen policies are evaluated on 10 held-out seeds (`7001`-`7010`) for each of five declared scenario families: stationary, seasonal, trend, shock and mixed. This is 50 episodes per model and 250 episodes overall.

In [ ]:
# Run fresh validation after training/export if desired:
RUN_VALIDATION = False
if RUN_VALIDATION:
    import subprocess, sys
    subprocess.run([sys.executable, "evaluate_all.py", "--roll-number", ROLL_NUMBER], check=True)

# Load the supplied final validation results.
results_path = Path("validation_results.csv")
if not results_path.exists():
    # If this notebook is in the project root and the supplied CSV is elsewhere, adjust this path.
    results_path = Path("final_submission_pack/validation_results.csv")
results = pd.read_csv(results_path)
results.head()

### Final local-validation comparison
These are the supplied final results from all 50 held-out episodes per technique.

In [ ]:
overall = (results.groupby("technique")["episode_cost"]
           .agg(["mean","std","median","min","max"])
           .sort_values("mean"))
overall.round(2)

In [ ]:
scenario_table = results.pivot_table(
    index="technique", columns="scenario", values="episode_cost", aggfunc="mean"
)
scenario_table.round(2)

**Observed ranking:** Double DQN is best (`97,134.15` mean cost), followed by PPO (`194,605.40`), A2C (`241,688.80`), Tabular Q-Learning (`788,377.80`), and REINFORCE (`1,063,956.40`). Double DQN is best in every scenario family.

## 9. Essential plots and convergence evidence

In [ ]:
ordered = overall["mean"].sort_values()
ax = ordered.plot(kind="barh", figsize=(7,3.2), title="Held-out validation: overall mean episode cost")
ax.set_xlabel("Mean episode cost (lower is better)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Validation-estimate convergence: cumulative mean as more held-out episodes are included.
# This is deliberately labeled as validation convergence, not a fabricated training learning curve.
fig, ax = plt.subplots(figsize=(8,4))
for technique, g in results.groupby("technique"):
    g = g.reset_index(drop=True)
    running = g["episode_cost"].expanding().mean()
    ax.plot(np.arange(1, len(g)+1), running, label=technique)
ax.set_xlabel("Number of held-out validation episodes included")
ax.set_ylabel("Cumulative mean episode cost")
ax.set_title("Convergence of held-out validation estimate")
ax.legend()
plt.tight_layout()
plt.show()

The cumulative validation means stabilize as more held-out episodes are included, and Double DQN remains well separated from the other techniques. This is convergence evidence for the *validation estimate*. No training-loss/return history was fabricated because the final training scripts did not persist intermediate learning-curve logs.

## 10. Export steps and submission mapping
Each `run_policy(observation)` performs inference only, uses deterministic action selection, and returns three actual order quantities in `{0,10,...,100}`. Model parameters are loaded once at module import.

In [ ]:
submission_map = pd.DataFrame([
    ["Tabular Q-Learning", "inventory_tabq_v1", "submissions/policy_tabular_q.py", "submissions/tabular_q.npz"],
    ["REINFORCE + baseline", "inventory_reinforce_v1", "submissions/policy_reinforce.py", "submissions/reinforce.pt"],
    ["A2C", "inventory_a2c_v1", "submissions/policy_a2c.py", "submissions/a2c.pt"],
    ["PPO", "inventory_ppo_v1", "submissions/policy_ppo.py", "submissions/ppo.pt"],
    ["Double DQN", "inventory_ddqn_v1", "submissions/policy_double_dqn.py", "submissions/double_dqn.pt"],
], columns=["Technique","Leaderboard submission identifier","Policy file","Model artifact"])
submission_map

In [ ]:
# Optional: verify that all five frozen submission policies import and produce valid deterministic actions.
def load_policy(path):
    path = Path(path)
    spec = importlib.util.spec_from_file_location("check_" + path.stem, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.run_policy

for _, row in submission_map.iterrows():
    policy_path = Path(row["Policy file"])
    artifact_path = Path(row["Model artifact"])
    print(row["Technique"], "policy:", policy_path.exists(), "artifact:", artifact_path.exists())

## 11. Final observations

1. **Double DQN** was the strongest and most robust approach. The factorized `3 x 11` Q representation reduced the action-learning burden dramatically compared with a 1,331-way joint output.
2. **PPO** was the strongest policy-gradient method, outperforming A2C across the combined validation set.
3. **A2C** learned a useful policy but at higher cost than PPO and Double DQN.
4. **REINFORCE** remained sample-inefficient and high variance despite a learned baseline.
5. **Tabular Q-Learning** suffered from discretization/state-coverage limitations and high held-out variability.
6. Inventory-aware derived features were important: strong policies must account for current inventory *and* outstanding pipeline stock relative to lead-time demand.
7. The final implementation executes exactly the action sampled during policy-gradient training; it does not alter the action after sampling, preserving correct log-probability updates.